# Instructor Solution - HW1 Seeded Circuits, Bit Flips, and Measurement Mapping

This notebook demonstrates the expected solution workflow and uses the hidden reference functions from `hw1_reference.py`.

In the private repository, students should receive the student template and instruction sheet. The instructor keeps `hw1_reference.py`, schema files, and grader logic hidden or instructor-only.

In [ ]:
%pip -q install qiskit qiskit-aer matplotlib jsonschema

In [ ]:
import json
from pathlib import Path
from qiskit.visualization import plot_histogram

# If this notebook is in the same folder as hw1_reference.py:
from hw1_reference import (
    generate_config,
    expected_display_bitstring,
    expected_logical_q_string,
    build_reference_circuit,
    simulate_reference_counts,
    reference_answers,
    validate_answers,
)

STUDENT_ID = "demo_student"
ASSIGNMENT_ID = "HW1"
SHOTS = 2048

config = generate_config(STUDENT_ID, ASSIGNMENT_ID)
print(json.dumps(config, indent=2))

In [ ]:
qc = build_reference_circuit(config)
qc.draw("mpl")

In [ ]:
expected_display = expected_display_bitstring(config)
expected_logical = expected_logical_q_string(config)
counts = simulate_reference_counts(config, shots=SHOTS)
dominant_bitstring = max(counts, key=counts.get)

print("Expected logical q[n-1]...q[0]:", expected_logical)
print("Expected displayed c[n-1]...c[0]:", expected_display)
print("Counts:", counts)
print("Dominant:", dominant_bitstring)
plot_histogram(counts)

In [ ]:
answers = reference_answers(STUDENT_ID, shots=SHOTS, assignment_id=ASSIGNMENT_ID)
answers["counts"] = {k: int(v) for k, v in counts.items()}
answers["reflection_bit_order"] = (
    "The measurement map determines which qubit value is stored in each classical bit. "
    "Qiskit displays count strings as c[n-1]...c[0], so the displayed string may be reversed "
    "relative to q[0]...q[n-1] if the measurement map is reversed."
)
answers["reflection_bit_flip"] = (
    "The final qubit bits are computed as initial_bits XOR flip_mask. "
    "Each 1 in the flip mask corresponds to an X gate that toggles that qubit."
)

with open("answers_solution.json", "w") as f:
    json.dump(answers, f, indent=2)

passed, feedback = validate_answers(answers, STUDENT_ID, shots=SHOTS)
print("Passed:", passed)
print("Feedback:")
for item in feedback:
    print("-", item)
print(json.dumps(answers, indent=2))

In [ ]:
# Optional: save the reference circuit as QPY for circuit-object validation.
from qiskit import qpy
with open("circuit_solution.qpy", "wb") as f:
    qpy.dump(qc, f)
print("Saved circuit_solution.qpy")

## What hidden reference generation means

The instructor does not need to manually store every student's expected bitstring. Instead, the autograder calls `generate_config(student_id)` and recomputes the expected circuit behavior from the same seed. This allows personalized assignments while keeping grading deterministic.

For HW1, the hidden reference functions check:

- correct seed and number of qubits,
- correct initial bits,
- correct bit-flip mask,
- correct final bits,
- correct measurement map,
- correct expected displayed bitstring,
- simulator counts dominated by the expected bitstring,
- sufficient reflection text.